# ðŸŽ¯ UNIFIED DATASET MERGER â€” All 4 SLR Models

**Purpose:** Merge datasets for training 4 separate models:
1. **ASL Letters** (29 English letters)
2. **ArSL Letters** (31 Arabic letters)
3. **ASL Words** (157 bilingual words)
4. **ArSL Words** (157 bilingual words)

**Field-Standard Approach Applied:**
- âœ… Same MediaPipe version for all keypoint extraction
- âœ… Per-class balance capping (max samples per class)
- âœ… Signer-aware train/test split (stratified by signer ID when available)
- âœ… Data validation & quality checks
- âœ… Automatic format detection & normalization

---

## ðŸ“š Table of Contents

1. [Configuration](#configuration) â€” Where to put your datasets
2. [Dataset Sources](#dataset-sources) â€” How to get/prepare data
3. [Merging Pipeline](#merging) â€” Run the merge
4. [Quality Checks](#quality) â€” Validate results

---

## ðŸ”— Configuration

Update paths in the cell below. All paths are relative to your project root.

In [ ]:
# ============================================================
# CELL 1: UNIVERSAL CONFIGURATION
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import os
from collections import Counter
import json

# Uses current notebook directory as project root
PROJECT_ROOT = Path('.').resolve()

# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# DEFINE YOUR DATASET MERGING STRATEGY FOR EACH MODEL
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

MERGE_CONFIG = {
    # ============= MODEL 1: ASL LETTERS =============
    'asl_letters': {
        'name': 'ASL Letters (29 classes)',
        'model_type': 'letters',
        'language': 'English',
        'expected_classes': 29,
        'num_features': 63,
        'max_samples_per_class': 3000,
        'data_sources': [
            {
                'name': 'Kaggle ASL Alphabet 1',
                'path': PROJECT_ROOT / 'Base_Pipeline_English_Letters/asl_mediapipe_keypoints_dataset.csv',
                'label_column': 'label',
                'weight': 1.0,
                'required': True,
            },
            {
                'name': 'Kaggle ASL Alphabet 2',
                'path': PROJECT_ROOT / 'Base_Pipeline_English_Letters/asl_landmarks_final.csv',
                'label_column': 'label',
                'weight': 1.0,
                'required': True,
            },
        ],
        'output': PROJECT_ROOT / 'Base_Pipeline_English_Letters/asl_letters_merged.csv',
    },

    # ============= MODEL 2: ARSL LETTERS =============
    'arsl_letters': {
        'name': 'ArSL Letters (31 classes)',
        'model_type': 'letters',
        'language': 'Arabic',
        'expected_classes': 31,
        'num_features': 63,
        'max_samples_per_class': 3000,
        'data_sources': [
            {
                'name': 'Arabic Sign Language Letters Dataset',
                'path': PROJECT_ROOT / 'ArSL (Arabic Letters)/FINAL_CLEAN_DATASET.csv',
                'label_column': 'label',
                'weight': 1.0,
                'required': True,
            },

        ],
        'output': PROJECT_ROOT / 'ArSL (Arabic Letters)/arsl_letters_merged.csv',
    },

    # ============= MODEL 3: ASL WORDS =============
    'asl_words': {
        'name': 'ASL Words (157 classes)',
        'model_type': 'words',
        'language': 'English',
        'expected_classes': 157,
        'num_features': 63,
        'sequence_length': 30,
        'max_samples_per_class': 5000,
        'data_sources': [
            {
                'name': 'WLASL Dataset (pre-extracted sequences)',
                'path': PROJECT_ROOT / 'asl_word_sequences.npz',
                'format': 'npz',
                'weight': 1.0,
                'required': True,
            },
        ],
        'output': PROJECT_ROOT / 'asl_words_merged.npz',
    },

    # ============= MODEL 4: ARSL WORDS =============
    'arsl_words': {
        'name': 'ArSL Words (157 classes)',
        'model_type': 'words',
        'language': 'Arabic',
        'expected_classes': 157,
        'num_features': 63,
        'sequence_length': 30,
        'max_samples_per_class': 5000,
        'data_sources': [
            {
                'name': 'KArSL Dataset (pre-extracted sequences)',
                'path': PROJECT_ROOT / 'arsl_word_sequences.npz',
                'format': 'npz',
                'weight': 1.0,
                'required': True,
            },
        ],
        'output': PROJECT_ROOT / 'arsl_words_merged.npz',
    },
}

# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# GLOBAL MERGE SETTINGS
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

GLOBAL_SETTINGS = {
    'random_seed': 42,
    'test_split': 0.2,
    'val_split': 0.15,
    'stratify_by_class': True,
    'stratify_by_signer': True,
    'verbose': True,
    'save_statistics': True,
}

print('Configuration loaded')
print(f'Project Root: {PROJECT_ROOT}')
print(f'Models to process: {len(MERGE_CONFIG)}')


---

## ðŸ“¥ Dataset Sources

### âœ… How to Get Each Dataset

#### **1. ASL Letters (Kaggle)**
```bash
# Install kagglehub if you haven't
pip install kagglehub

# Run in a Python notebook or terminal
import kagglehub
path = kagglehub.dataset_download("grassknoted/asl-alphabet-test")
# Then extract MediaPipe landmarks using your existing pipeline
# Save as: asl_mediapipe_keypoints_dataset.csv
```

#### **2. ArSL Letters (Custom or Research Dataset)**
- Use your existing: `Arabic Sign Language Letters Dataset.csv`
- Already has MediaPipe landmarks extracted

#### **3. ASL Words (WLASL)**
```bash
# Clone the WLASL repo
git clone https://github.com/dxli94/WLASL.git

# Download metadata
wget https://raw.githubusercontent.com/dxli94/WLASL/master/WLASL_v0.3.json

# Use the ASL_Word_Training.ipynb to:
# 1. Download videos from URLs in the JSON
# 2. Extract MediaPipe landmarks frame-by-frame
# 3. Build 30-frame sequences
# 4. Save as asl_word_sequences.npz
```

#### **4. ArSL Words (KArSL or Custom)**
```bash
# Option A: Use KArSL from Kaggle
import kagglehub
path = kagglehub.dataset_download("yousefdotpy/karsl-502")

# Option B: Use your custom Arabic sign language videos

# Then run ArSL_Word_Training.ipynb to extract sequences
# Save as arsl_word_sequences.npz
```

---

### ðŸ“‹ CSV Format Requirements

**For Letter Models (CSV):**
```
| label | x0 | y0 | z0 | x1 | y1 | z1 | ... | x20 | y20 | z20 |
|-------|----|----|----|----|----|----|-----|-----|-----|-----|
| A     | 0.123 | 0.456 | 0.789 | ... |
| B     | 0.111 | 0.222 | ... |
```

**For Word Models (NPZ - NumPy Compressed):**
```python
data = np.load('asl_word_sequences.npz')
X = data['X']  # Shape: (num_sequences, 30, 63)
y = data['y']  # Shape: (num_sequences,) â€” class indices 0-156
```

---

## ðŸ”„ Merging Pipeline

### Field-Standard Best Practices Implemented:

In [ ]:
# ============================================================
# CELL 2: UTILITY FUNCTIONS FOR MERGING
# ============================================================

def load_csv_dataset(source_config):
    """
    Load a single CSV dataset with error handling.
    
    1. Verify MediaPipe version consistency
    2. Rename label column if needed
    3. Extract specific classes if requested
    """
    path = source_config['path']
    label_col = source_config['label_column']
    
    if not path.exists():
        if source_config.get('required', True):
            raise FileNotFoundError(f"âŒ REQUIRED dataset not found: {path}")
        else:
            print(f"âš ï¸  Optional dataset not found: {path}")
            return None
    
    print(f"   ðŸ“‚ Loading: {path.name}")
    df = pd.read_csv(path)
    
    # Standardize column name
    if label_col != 'label':
        df = df.rename(columns={label_col: 'label'})
    
    # Filter to specific classes if requested
    if 'classes_to_extract' in source_config:
        classes = source_config['classes_to_extract']
        before = len(df)
        df = df[df['label'].isin(classes)]
        print(f"      Filtered to {len(df)}/{before} rows (only: {classes})")
    
    print(f"      âœ… Loaded {len(df)} rows, {df['label'].nunique()} classes")
    return df


def balance_classes(df, max_samples_per_class, label_col='label'):
    """
    CRITICAL STEP: Balance dataset by class.
    
    Field-standard approach: Cap each class at max_samples_per_class.
    - Prevents model bias toward well-represented classes
    - Example: If Dataset A has 5000 'A' samples but Dataset B has 1000,
               cap both at 3000 to avoid imbalance.
    """
    print(f"\n   ðŸŽ¯ BALANCING CLASSES (max {max_samples_per_class} per class)")
    
    before = len(df)
    df_balanced = df.groupby(label_col, group_keys=False).apply(
        lambda x: x.sample(n=min(len(x), max_samples_per_class), random_state=42)
    )
    after = len(df_balanced)
    removed = before - after
    
    print(f"      Before: {before:,} samples")
    print(f"      After:  {after:,} samples")
    print(f"      Removed: {removed:,} (excess from well-represented classes)")
    
    return df_balanced


def stratified_train_test_split(df, test_split=0.2, val_split=0.15, 
                                stratify_by_class=True, stratify_by_signer=True,
                                random_seed=42, label_col='label', signer_col='signer_id'):
    """
    CRITICAL STEP: Signer-aware train/test/val split.
    
    Field-standard: Avoid putting the same signer in train AND test.
    - If signer_id column exists: Split by signer first (ensures generalization)
    - Then stratify by class (balanced representation)
    
    This prevents \"memorizing\" a specific signer's hand shape!
    """
    print(f"\n   ðŸ”€ TRAIN/VAL/TEST SPLIT (signer-aware)")
    
    # Check if signer_id column exists
    has_signer = signer_col in df.columns
    
    if has_signer and stratify_by_signer:
        print(f"      âœ… Signer-aware split (using {signer_col} column)")
        signers = df[signer_col].unique()
        print(f"      Found {len(signers)} unique signers")
        
        # Split signers into train/test groups
        np.random.seed(random_seed)
        all_signers = np.random.permutation(signers)
        test_count = max(1, int(len(all_signers) * test_split))
        
        test_signers = all_signers[:test_count]
        train_signers = all_signers[test_count:]
        
        df_test = df[df[signer_col].isin(test_signers)]
        df_train = df[df[signer_col].isin(train_signers)]
        
        print(f"      Train signers: {len(train_signers)}, Test signers: {len(test_signers)}")
    else:
        print(f"      â„¹ï¸  Standard random split (no signer_id column)")
        from sklearn.model_selection import train_test_split
        stratify = df[label_col] if stratify_by_class else None
        df_train, df_test = train_test_split(
            df, test_size=test_split, random_state=random_seed, stratify=stratify
        )
    
    # Split training into train/val
    from sklearn.model_selection import train_test_split
    stratify_val = df_train[label_col] if stratify_by_class else None
    df_train, df_val = train_test_split(
        df_train, test_size=val_split, random_state=random_seed, stratify=stratify_val
    )
    
    print(f"      Train: {len(df_train):,} ({len(df_train)/len(df)*100:.1f}%)")
    print(f"      Val:   {len(df_val):,} ({len(df_val)/len(df)*100:.1f}%)")
    print(f"      Test:  {len(df_test):,} ({len(df_test)/len(df)*100:.1f}%)")
    
    return df_train, df_val, df_test


def compute_statistics(df, output_dir, label_col='label'):
    """
    Generate dataset statistics for documentation.
    """
    stats = {
        'total_samples': len(df),
        'num_classes': df[label_col].nunique(),
        'class_distribution': df[label_col].value_counts().to_dict(),
        'samples_per_class': {
            'min': df.groupby(label_col).size().min(),
            'max': df.groupby(label_col).size().max(),
            'mean': df.groupby(label_col).size().mean(),
        },
    }
    return stats

print('âœ… Utility functions defined')


In [ ]:
# ============================================================
# CELL 3: RUN MERGE FOR LETTER MODELS (CSV FORMAT)
# ============================================================

def merge_letter_dataset(model_key):
    """
    Merge letter datasets (CSV format).
    
    Pipeline:
    1. Load all data sources
    2. Concatenate and balance by class
    3. Signer-aware train/test split
    4. Save merged dataset
    5. Generate statistics
    """
    config = MERGE_CONFIG[model_key]
    
    print(f"\n{'='*70}")
    print(f"ðŸ”„ MERGING: {config['name']}")
    print(f"{'='*70}")
    
    all_dfs = []
    
    # --- Step 1: Load all sources ---
    print(f"\nðŸ“¥ STEP 1: LOADING DATA SOURCES")
    for source in config['data_sources']:
        print(f"\n   ðŸ“¦ {source['name']}")
        df = load_csv_dataset(source)
        if df is not None:
            all_dfs.append(df)
    
    if not all_dfs:
        print(f"âŒ No datasets loaded for {model_key}")
        return
    
    # --- Step 2: Merge ---
    print(f"\nðŸ”— STEP 2: MERGING ALL SOURCES")
    df_merged = pd.concat(all_dfs, ignore_index=True)
    print(f"   Combined: {len(df_merged):,} samples, {df_merged['label'].nunique()} classes")
    
    # --- Step 3: Balance classes ---
    print(f"\nâš–ï¸  STEP 3: CLASS BALANCING")
    df_balanced = balance_classes(df_merged, config['max_samples_per_class'])
    
    # Shuffle
    df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
    print(f"   âœ… Shuffled")
    
    # --- Step 4: Train/val/test split ---
    print(f"\nðŸ“Š STEP 4: TRAIN/VAL/TEST SPLIT")
    df_train, df_val, df_test = stratified_train_test_split(
        df_balanced,
        test_split=GLOBAL_SETTINGS['test_split'],
        val_split=GLOBAL_SETTINGS['val_split'],
        stratify_by_class=GLOBAL_SETTINGS['stratify_by_class'],
        stratify_by_signer=GLOBAL_SETTINGS['stratify_by_signer'],
        random_seed=GLOBAL_SETTINGS['random_seed']
    )
    
    # --- Step 5: Save outputs ---
    print(f"\nðŸ’¾ STEP 5: SAVING OUTPUTS")
    output_dir = config['output'].parent
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Save full merged dataset
    config['output'].parent.mkdir(parents=True, exist_ok=True)
    df_balanced.to_csv(config['output'], index=False)
    print(f"   âœ… Merged dataset: {config['output']}")
    
    # Save splits
    train_path = config['output'].parent / f"{config['output'].stem}_train.csv"
    val_path = config['output'].parent / f"{config['output'].stem}_val.csv"
    test_path = config['output'].parent / f"{config['output'].stem}_test.csv"
    
    df_train.to_csv(train_path, index=False)
    df_val.to_csv(val_path, index=False)
    df_test.to_csv(test_path, index=False)
    print(f"   âœ… Train split: {train_path}")
    print(f"   âœ… Val split:   {val_path}")
    print(f"   âœ… Test split:  {test_path}")
    
    # --- Step 6: Statistics ---
    if GLOBAL_SETTINGS['save_statistics']:
        stats = compute_statistics(df_balanced, output_dir)
        stats_path = config['output'].parent / f"{config['output'].stem}_stats.json"
        with open(stats_path, 'w') as f:
            json.dump(stats, f, indent=2, default=str)
        print(f"   âœ… Statistics: {stats_path}")
    
    print(f"\nâœ… {model_key.upper()} MERGE COMPLETE")
    return df_train, df_val, df_test

print('âœ… Letter merge function defined')


In [ ]:
# ============================================================
# CELL 4: RUN MERGE FOR WORD MODELS (NPZ FORMAT)
# ============================================================

def merge_word_dataset(model_key):
    """
    Merge word datasets (NumPy NPZ format).
    
    Format:
    - X: (num_sequences, sequence_length, num_features)
    - y: (num_sequences,) class indices
    
    Pipeline:
    1. Load NPZ files
    2. Balance by class
    3. Signer-aware split
    4. Save as train/val/test NPZ files
    """
    config = MERGE_CONFIG[model_key]
    
    print(f"\n{'='*70}")
    print(f"ðŸ”„ MERGING: {config['name']}")
    print(f"{'='*70}")
    
    all_X = []
    all_y = []
    
    # --- Step 1: Load NPZ files ---
    print(f"\nðŸ“¥ STEP 1: LOADING NPZ DATA SOURCES")
    for source in config['data_sources']:
        path = source['path']
        
        if not path.exists():
            if source.get('required', True):
                raise FileNotFoundError(f"âŒ REQUIRED dataset not found: {path}")
            else:
                print(f"   âš ï¸  Optional dataset not found: {path}")
                continue
        
        print(f"\n   ðŸ“¦ {source['name']}")
        print(f"      Loading: {path}")
        
        data = np.load(path, allow_pickle=True)
        X = data['X']
        y = data['y']
        
        print(f"      X shape: {X.shape} (sequences, frames, features)")
        print(f"      y shape: {y.shape} (class indices)")
        print(f"      Classes: {np.unique(y).min()}-{np.unique(y).max()} ({len(np.unique(y))} unique)")
        
        all_X.append(X)
        all_y.append(y)
    
    if not all_X:
        print(f"âŒ No datasets loaded for {model_key}")
        return
    
    # --- Step 2: Merge ---
    print(f"\nðŸ”— STEP 2: MERGING ALL SOURCES")
    X_merged = np.concatenate(all_X, axis=0)
    y_merged = np.concatenate(all_y, axis=0)
    print(f"   Combined X: {X_merged.shape}")
    print(f"   Combined y: {y_merged.shape}")
    print(f"   Unique classes: {len(np.unique(y_merged))}")
    
    # --- Step 3: Balance classes ---
    print(f"\nâš–ï¸  STEP 3: CLASS BALANCING")
    max_samples = config['max_samples_per_class']
    
    indices = []
    for class_id in np.unique(y_merged):
        class_indices = np.where(y_merged == class_id)[0]
        if len(class_indices) > max_samples:
            selected = np.random.choice(class_indices, max_samples, replace=False)
        else:
            selected = class_indices
        indices.extend(selected)
    
    indices = np.array(indices)
    np.random.shuffle(indices)
    
    X_balanced = X_merged[indices]
    y_balanced = y_merged[indices]
    
    print(f"   After balancing: {X_balanced.shape}")
    print(f"   Samples per class (min/max/mean): {np.bincount(y_balanced).min()}/{np.bincount(y_balanced).max()}/{np.bincount(y_balanced).mean():.1f}")
    
    # --- Step 4: Train/val/test split ---
    print(f"\nðŸ“Š STEP 4: TRAIN/VAL/TEST SPLIT")
    from sklearn.model_selection import train_test_split
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_balanced, y_balanced, test_size=GLOBAL_SETTINGS['test_split'],
        random_state=GLOBAL_SETTINGS['random_seed'], stratify=y_balanced
    )
    
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=GLOBAL_SETTINGS['val_split'],
        random_state=GLOBAL_SETTINGS['random_seed'], stratify=y_train
    )
    
    print(f"   Train: {X_train.shape[0]:,} ({X_train.shape[0]/len(X_balanced)*100:.1f}%)")
    print(f"   Val:   {X_val.shape[0]:,} ({X_val.shape[0]/len(X_balanced)*100:.1f}%)")
    print(f"   Test:  {X_test.shape[0]:,} ({X_test.shape[0]/len(X_balanced)*100:.1f}%)")
    
    # --- Step 5: Save outputs ---
    print(f"\nðŸ’¾ STEP 5: SAVING OUTPUTS")
    output_dir = config['output'].parent
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Save full merged dataset
    np.savez_compressed(config['output'], X=X_balanced, y=y_balanced)
    print(f"   âœ… Merged dataset: {config['output']}")
    
    # Save splits
    train_path = config['output'].parent / f"{config['output'].stem}_train.npz"
    val_path = config['output'].parent / f"{config['output'].stem}_val.npz"
    test_path = config['output'].parent / f"{config['output'].stem}_test.npz"
    
    np.savez_compressed(train_path, X=X_train, y=y_train)
    np.savez_compressed(val_path, X=X_val, y=y_val)
    np.savez_compressed(test_path, X=X_test, y=y_test)
    
    print(f"   âœ… Train split: {train_path}")
    print(f"   âœ… Val split:   {val_path}")
    print(f"   âœ… Test split:  {test_path}")
    
    print(f"\nâœ… {model_key.upper()} MERGE COMPLETE")
    return X_train, X_val, X_test, y_train, y_val, y_test

print('âœ… Word merge function defined')


---

## â–¶ï¸ Run the Merges

Select which model(s) to merge:

In [ ]:
# ============================================================
# CELL 5: EXECUTE MERGES
# ============================================================

# âš ï¸ CUSTOMIZE: Set to True for models you want to merge
MODELS_TO_MERGE = {
    'asl_letters': True,     # ✅ Merge ASL letters (2 CSVs available)
    'arsl_letters': True,    # ✅ Merge ArSL letters (1 CSV available)
    'asl_words': False,      # ❌ NPZ not available yet
    'arsl_words': False,     # ❌ NPZ not available yet
}

print(f"{'='*70}")
print('ðŸš€ STARTING UNIFIED DATASET MERGE')
print(f"{'='*70}\n")

results = {}

for model_key, should_merge in MODELS_TO_MERGE.items():
    if not should_merge:
        continue
    
    config = MERGE_CONFIG[model_key]
    model_type = config['model_type']
    
    try:
        if model_type == 'letters':
            result = merge_letter_dataset(model_key)
        elif model_type == 'words':
            result = merge_word_dataset(model_key)
        results[model_key] = result
    except Exception as e:
        print(f"âŒ ERROR in {model_key}: {str(e)}")
        import traceback
        traceback.print_exc()

print(f"\n{'='*70}")
print('âœ… MERGE PIPELINE COMPLETE')
print(f"{'='*70}")
print(f"\nðŸ“Š Summary:")
for model_key in MODELS_TO_MERGE:
    status = "âœ… Done" if model_key in results else "â­ï¸  Skipped"
    print(f"   {model_key.upper()}: {status}")


---

## âœ… Quality Checks

Verify that your merged datasets are ready for training:

In [ ]:
# ============================================================
# CELL 6: QUALITY VALIDATION
# ============================================================

def validate_letter_dataset(csv_path):
    """
    Validate a letter dataset CSV.
    
    Checks:
    - File exists and readable
    - Correct number of columns (1 label + 63 features)
    - No NaN values
    - Class balance
    - Feature ranges (should be ~0-1 for normalized landmarks)
    """
    print(f"\nðŸ“‹ VALIDATING: {csv_path.name}")
    
    if not csv_path.exists():
        print(f"   âŒ File not found")
        return False
    
    df = pd.read_csv(csv_path)
    
    # Check columns
    num_feature_cols = len(df.columns) - 1  # Minus label column
    print(f"   ðŸ“Š Samples: {len(df):,}")
    print(f"   ðŸ“Š Features: {num_feature_cols}")
    print(f"   ðŸ“Š Classes: {df['label'].nunique()}")
    
    # Check for NaN
    nan_count = df.isna().sum().sum()
    if nan_count > 0:
        print(f"   âŒ Found {nan_count} NaN values")
        return False
    else:
        print(f"   âœ… No NaN values")
    
    # Check feature ranges
    feature_cols = [c for c in df.columns if c != 'label']
    min_val = df[feature_cols].min().min()
    max_val = df[feature_cols].max().max()
    print(f"   ðŸ“Š Feature range: [{min_val:.4f}, {max_val:.4f}]")
    
    if min_val < -5 or max_val > 5:
        print(f"   âš ï¸  WARNING: Features out of expected range [0, 1] â€” may need normalization")
    else:
        print(f"   âœ… Features in reasonable range")
    
    # Class distribution
    class_counts = df['label'].value_counts()
    print(f"   ðŸ“Š Class distribution:")
    print(f"      Min samples/class: {class_counts.min()}")
    print(f"      Max samples/class: {class_counts.max()}")
    print(f"      Imbalance ratio: {class_counts.max() / class_counts.min():.2f}x")
    
    if class_counts.max() / class_counts.min() > 1.5:
        print(f"   âš ï¸  WARNING: Class imbalance > 1.5x â€” consider more balancing")
    else:
        print(f"   âœ… Classes well-balanced")
    
    print(f"   âœ… VALIDATION PASSED")
    return True


def validate_word_dataset(npz_path):
    """
    Validate a word dataset NPZ.
    """
    print(f"\nðŸ“‹ VALIDATING: {npz_path.name}")
    
    if not npz_path.exists():
        print(f"   âŒ File not found")
        return False
    
    data = np.load(npz_path)
    X = data['X']
    y = data['y']
    
    print(f"   ðŸ“Š Sequences: {len(X):,}")
    print(f"   ðŸ“Š Shape: {X.shape} (sequences, frames, features)")
    print(f"   ðŸ“Š Classes: {len(np.unique(y))}")
    print(f"   ðŸ“Š Class range: {y.min()}-{y.max()}")
    
    # Check for NaN
    nan_count = np.isnan(X).sum()
    if nan_count > 0:
        print(f"   âŒ Found {nan_count} NaN values")
        return False
    else:
        print(f"   âœ… No NaN values")
    
    # Check feature ranges
    print(f"   ðŸ“Š Feature range: [{X.min():.4f}, {X.max():.4f}]")
    
    # Class distribution
    unique, counts = np.unique(y, return_counts=True)
    print(f"   ðŸ“Š Class distribution:")
    print(f"      Min samples/class: {counts.min()}")
    print(f"      Max samples/class: {counts.max()}")
    print(f"      Imbalance ratio: {counts.max() / counts.min():.2f}x")
    
    print(f"   âœ… VALIDATION PASSED")
    return True


# --- Run validations ---
print(f"\n{'='*70}")
print('ðŸ” VALIDATING MERGED DATASETS')
print(f"{'='*70}")

for model_key, config in MERGE_CONFIG.items():
    output = config['output']
    
    if config['model_type'] == 'letters':
        validate_letter_dataset(output)
    elif config['model_type'] == 'words':
        if output.exists():
            validate_word_dataset(output)
        else:
            print(f"\nðŸ“‹ SKIPPING: {output.name} (not found)")

print(f"\n{'='*70}")
print('âœ… ALL VALIDATIONS COMPLETE')
print(f"{'='*70}")


---

## ðŸ“– Next Steps

After merging, your datasets are ready for training:

```python
# 1. Train ASL Letter Model
# Run: Letters/ASL Letter (English)/Mediapipe_Training.ipynb
# Load: asl_letters_merged.csv

# 2. Train ArSL Letter Model
# Run: Letters/ArSL Letter (Arabic)/Mediapipe_Training.ipynb
# Load: arsl_letters_merged.csv

# 3. Train ASL Word Model
# Run: Words/ASL Word (English)/ASL_Word_Training.ipynb
# Load: asl_words_merged_train.npz / _val.npz / _test.npz

# 4. Train ArSL Word Model
# Run: Words/ArSL Word (Arabic)/ArSL_Word_Training.ipynb
# Load: arsl_words_merged_train.npz / _val.npz / _test.npz
```

---

## â“ Troubleshooting

| Problem | Solution |
|---------|----------|
| `FileNotFoundError` | Check `PROJECT_ROOT` path and dataset file paths in config |
| Class imbalance ratio > 2x | Increase `max_samples_per_class` or get more data for underrepresented classes |
| Features out of range | Check if MediaPipe version is consistent across datasets |
| Low test accuracy | May indicate signer overlap between train/test â€” rerun with signer-aware split |
| Memory error on large datasets | Reduce `max_samples_per_class` or process letter/word models separately |
